# ProverbsLM — LoRA Fine-Tuning on Your Codebase

**Goal:** Fine-tune `qwen2.5-coder-7b` on all 50 RAXX apps to create a model that
knows your exact coding style — Next.js, Prisma, Stripe, Capacitor, Codemagic, everything.

## Infrastructure options (pick one)
- **RunPod A100 80GB**: $1.99/hr → ~6-8h = ~$16 total
- **Google Colab Pro A100**: $10/month subscription
- **Vast.ai RTX 4090**: $0.50/hr → ~12h = ~$6 total

## What LoRA does
LoRA (Low-Rank Adaptation) fine-tunes only 0.1% of a model's parameters.
It takes qwen2.5-coder-7b (which already knows how to code) and teaches it YOUR patterns.
After merging, you get a 7B model that builds apps exactly the way you do.

## After this notebook
1. Download `proverbs-7b-merged.gguf`
2. Place in `~/.proverbs/models/`
3. Server auto-detects it on next restart
4. Run `proverbs` — it generates code in your style, no internet needed

In [ ]:
# ── 1. Check GPU ─────────────────────────────────────────────────────────────
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip() if r.returncode == 0 else 'No GPU!')

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 2. Install unsloth (2x faster LoRA training) ─────────────────────────────
# unsloth supports qwen2.5, llama3, mistral, deepseek, etc.
!pip install unsloth -q
!pip install transformers datasets tqdm -q

# For GGUF export
!pip install llama-cpp-python -q

print('Dependencies ready.')

In [ ]:
# ── 3. Upload training data ───────────────────────────────────────────────────
# You need to upload these files from your Mac:
#   ~/.proverbs/pretrain_data/raxx_codebase.jsonl   (your 50 apps)
#   ~/.proverbs/pretrain_data/CodeAlpaca-20k.jsonl  (general code)
#   ~/.proverbs/pretrain_data/code_instructions_122k_alpaca_style.jsonl
#   ~/.proverbs/pretrain_data/python_code_instructions_18k_alpaca.jsonl
#   ~/.proverbs/sessions/*.jsonl  (your actual Claude sessions)

# Option A: Upload via Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/proverbs_training/'

# Option B: Direct upload
from google.colab import files
import os

DATA_DIR = '/content/training_data/'
os.makedirs(DATA_DIR, exist_ok=True)

print('Upload your training data files (all .jsonl files from ~/.proverbs/pretrain_data/ and ~/.proverbs/sessions/)')
print('TIP: Zip them first: cd ~/.proverbs && zip -r training_data.zip pretrain_data/ sessions/')

uploaded = files.upload()
for fname, data in uploaded.items():
    if fname.endswith('.zip'):
        import zipfile
        with zipfile.ZipFile(fname) as z:
            z.extractall(DATA_DIR)
        print(f'Extracted {fname}')
    else:
        with open(os.path.join(DATA_DIR, fname), 'wb') as f:
            f.write(data)

# Count examples
total = 0
for root, _, files_list in os.walk(DATA_DIR):
    for f in files_list:
        if f.endswith('.jsonl'):
            n = sum(1 for _ in open(os.path.join(root, f)))
            print(f'  {f}: {n:,} examples')
            total += n
print(f'\nTotal: {total:,} training examples')

In [ ]:
# ── 4. Load base model with unsloth ──────────────────────────────────────────
from unsloth import FastLanguageModel
import torch

# Model options (pick based on your VRAM):
# A100 80GB → qwen2.5-coder-7b (best) or qwen2.5-coder-14b (if feeling ambitious)
# A100 40GB → qwen2.5-coder-7b
# T4 16GB   → qwen2.5-coder-3b

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"  # Change to -14B on A100 80GB
MAX_SEQ_LEN = 4096   # Increase to 8192 if you have enough VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name   = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype        = None,   # Auto-detect (bfloat16 on A100)
    load_in_4bit = True,   # 4-bit quantization for memory efficiency
)

print(f'Model loaded: {MODEL_NAME}')
print(f'Parameters:   {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B')

In [ ]:
# ── 5. Add LoRA adapters ──────────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r              = 32,     # LoRA rank — higher = more capacity but more VRAM
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 32,     # Scaling factor (usually == r)
    lora_dropout   = 0,      # 0 for unsloth optimized kernels
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable/1e6:.1f}M / {total/1e9:.1f}B = {100*trainable/total:.2f}%')
print('LoRA adapters added — training only the important parts.')

In [ ]:
# ── 6. Prepare training dataset ───────────────────────────────────────────────
import json, os, random
from datasets import Dataset

def load_all_jsonl(data_dir: str) -> list:
    examples = []
    for root, _, files_list in os.walk(data_dir):
        for fname in files_list:
            if not fname.endswith('.jsonl'):
                continue
            fpath = os.path.join(root, fname)
            with open(fpath) as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        obj = json.loads(line)
                        # Handle messages format
                        if 'messages' in obj:
                            examples.append(obj)
                        # Handle text format — wrap as assistant response
                        elif 'text' in obj:
                            text = obj['text']
                            # Split instruction/response on newline boundary
                            lines = text.split('\n')
                            mid = len(lines) // 3
                            examples.append({'messages': [
                                {'role': 'system', 'content': 'You are Proverbs, an expert coding assistant for RAXX BEATS STUDIOS apps. Be concise and precise.'},
                                {'role': 'user', 'content': '\n'.join(lines[:mid]).strip() or 'Complete this code:'},
                                {'role': 'assistant', 'content': '\n'.join(lines[mid:]).strip()},
                            ]})
                    except Exception:
                        continue
    return examples

def apply_chat_template(example):
    messages = example['messages']
    # Use the tokenizer's built-in chat template
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

print('Loading training data...')
raw_examples = load_all_jsonl(DATA_DIR)
random.shuffle(raw_examples)

# Use 90% for training, 10% for validation
split = int(len(raw_examples) * 0.9)
train_data = raw_examples[:split]
valid_data = raw_examples[split:]

print(f'Train: {len(train_data):,}  Valid: {len(valid_data):,}')

train_ds = Dataset.from_list(train_data).map(apply_chat_template, remove_columns=['messages'])
valid_ds = Dataset.from_list(valid_data).map(apply_chat_template, remove_columns=['messages'])

# Filter by length
def is_valid_length(ex):
    return len(tokenizer.encode(ex['text'])) <= MAX_SEQ_LEN

train_ds = train_ds.filter(is_valid_length)
valid_ds = valid_ds.filter(is_valid_length)

print(f'After length filter — Train: {len(train_ds):,}  Valid: {len(valid_ds):,}')

In [ ]:
# ── 7. Train ──────────────────────────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments

# Estimate training time:
#   A100 80GB: ~2000 steps/hour  → 3000 steps = ~1.5 hours
#   A100 40GB: ~1200 steps/hour  → 3000 steps = ~2.5 hours
#   RTX 4090:  ~500 steps/hour   → 3000 steps = ~6 hours

trainer = SFTTrainer(
    model             = model,
    tokenizer         = tokenizer,
    train_dataset     = train_ds,
    eval_dataset      = valid_ds,
    dataset_text_field= "text",
    max_seq_length    = MAX_SEQ_LEN,
    dataset_num_proc  = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,    # effective batch = 8
        warmup_steps                = 100,
        max_steps                   = 3000, # ~3-6h depending on GPU
        learning_rate               = 2e-4,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 10,
        evaluation_strategy         = "steps",
        eval_steps                  = 200,
        save_strategy               = "steps",
        save_steps                  = 500,
        output_dir                  = "/content/proverbs-lora",
        optim                       = "adamw_8bit",
        lr_scheduler_type           = "cosine",
        report_to                   = "none",
    ),
)

print('Starting training...')
print('Watch the loss drop:')
print('  ~2.5 = start')
print('  ~1.5 = learning your style')
print('  ~1.0 = strong — knows your patterns well')
print('  ~0.7 = excellent — generates like you')
print()

trainer_stats = trainer.train()
print(f'\nTraining complete!')
print(f'Final loss: {trainer_stats.training_loss:.4f}')

In [ ]:
# ── 8. Merge LoRA weights into full model ─────────────────────────────────────
# This creates a standalone model (no adapter needed at inference)
from unsloth import FastLanguageModel

print('Merging LoRA weights into base model...')
model.save_pretrained_merged(
    "/content/proverbs-7b-merged",
    tokenizer,
    save_method = "merged_16bit",  # Full precision merged model
)
print('Merged model saved to /content/proverbs-7b-merged/')

In [ ]:
# ── 9. Export to GGUF for offline inference ───────────────────────────────────
# This creates the .gguf file that loads in your Proverbs server

print('Exporting to GGUF (Q4_K_M quantization — best quality/size balance)...')
model.save_pretrained_gguf(
    "/content/proverbs-7b",
    tokenizer,
    quantization_method = "q4_k_m",
)

import os, glob
gguf_files = glob.glob('/content/proverbs-7b*.gguf')
for f in gguf_files:
    size_gb = os.path.getsize(f) / 1e9
    print(f'GGUF ready: {f} ({size_gb:.1f} GB)')

In [ ]:
# ── 10. Test the model before downloading ─────────────────────────────────────
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)   # Enable 2x faster inference

test_prompts = [
    "Write a Next.js 15 server action that creates a new user in Prisma and sends a welcome email",
    "Write a Prisma schema for a subscription system with users, plans, and billing history",
    "Write a Stripe webhook handler for subscription created, updated, and cancelled events",
    "Write a Capacitor plugin call to take a photo and upload it to Supabase storage",
]

for prompt in test_prompts:
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs, max_new_tokens=300, temperature=0.2,
        do_sample=True, use_cache=True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    
    print(f'\n─── {prompt[:60]}... ───')
    print(response[:500])
    print()

In [ ]:
# ── 11. Download the GGUF file ────────────────────────────────────────────────
import glob
from google.colab import files

gguf_files = glob.glob('/content/proverbs-7b*.gguf')
if gguf_files:
    gguf = gguf_files[0]
    size_gb = os.path.getsize(gguf) / 1e9
    print(f'Downloading {gguf} ({size_gb:.1f} GB)...')
    print('NOTE: If the file is >2GB, use the Files panel on the left to download')
    files.download(gguf)
else:
    print('GGUF not found — check that cell 9 completed successfully')

print()
print('─' * 50)
print('On your Mac, place the file at:')
print('  ~/.proverbs/models/proverbs-7b-Q4_K_M.gguf')
print()
print('The server will auto-detect it on next start.')
print('Run: launchctl kickstart -k gui/$(id -u)/com.proverbs.server')
print()
print('Then in Proverbs CLI:')
print('  /backend auto   ← switches to your trained model')
print('  /model proverbs-7b-Q4_K_M')

## Continuous improvement

Every time you build a new app with Proverbs, the session gets saved to `~/.proverbs/sessions/`.
Re-run this notebook monthly with the new sessions to keep teaching the model your latest patterns.

**Roadmap:**
- Round 1: 7B model on current 50 apps → knows your full stack
- Round 2: Add App Store/Codemagic API sessions → knows your CI/CD flow  
- Round 3: Export Claude conversation history → learns from every answer you've gotten
- Round 4: 14B model on dedicated server → matches GPT-4 code quality
- Round 5: 34B model → exceeds GPT-4 on your specific stack